In [86]:
import pandas as pd
import torch

# split into train and test
from sklearn.model_selection import train_test_split

device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


In [87]:
# load csv
# ./data/digit-recognizer/train.csv

df = pd.read_csv("../data/digit-recognizer/train.csv")

In [88]:
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [89]:
# assert labels are values from 0 to 9
assert df["label"].isin(range(10)).all()
# assert pixels are values from 0 to 255
for col in df.columns[1:]:
    assert df[col].isin(range(256)).all()

# change values to boolean true if pixel value is greater than 128, false otherwise
for col in df.columns[1:]:
    df[col] = df[col] > 64
    assert df[col].isin([True, False]).all()

# chang columns to boolean
for col in df.columns[1:]:
    df[col] = df[col].astype(bool)
df

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41996,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41997,7,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41998,6,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [102]:
# create neural network model using torch
class DigitClassifier(torch.nn.Module):
    def __init__(self, input_size: int, hidden_layer_size: int, output_size: int)-> None:
        super(DigitClassifier, self).__init__()
        self.input_to_hidden_layer = torch.nn.Linear(input_size, hidden_layer_size)
        self.hidden_to_output_layer = torch.nn.Linear(hidden_layer_size, output_size)

    def forward(self, input_tensor: torch.Tensor) -> torch.Tensor:
        z1 = self.input_to_hidden_layer(input_tensor)
        a1 = torch.relu(z1)
        z2 = self.hidden_to_output_layer(a1)
        # a2 = torch.softmax(z2, dim=1)
        return z2


X_train, X_test, y_train, y_test = train_test_split(
    df.drop("label", axis=1), df["label"], test_size=0.2, random_state=42
)

# convert to torch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)
# create data loaders
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)

# Create the model
# input size is 784 (28x28 pixels), hidden size is 128, output size is 10 (digits from 0 to 9)
model = DigitClassifier(input_size=784, hidden_layer_size=128, output_size=10)
# train the model
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
# train the model for 1000 epochs
for epoch in range(100):
    # forward pass
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        # make predictions
        y_hat = model(X_batch)
        # calculate loss
        loss = criterion(y_hat, y_batch)
        # calculate gradients
        loss.backward()
        # update weights
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        accuracy = (y_hat.argmax(dim=1) == y_batch).sum().item() / len(y_batch)
        precision = (y_hat.argmax(dim=1)[y_batch == 1] == 1).sum().item() / (y_hat.argmax(dim=1) == 1).sum().item()
        recall = (y_hat.argmax(dim=1)[y_batch == 1] == 1).sum().item() / (y_batch == 1).sum().item()
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}, Accuracy: {accuracy:.6f}, Precision: {precision:.6f}, Recall: {recall:.6f}")



Epoch [10/100], Loss: 0.0697, Accuracy: 0.968750, Precision: 1.000000, Recall: 1.000000
Epoch [20/100], Loss: 0.0184, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [30/100], Loss: 0.0030, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [40/100], Loss: 0.0044, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [50/100], Loss: 0.0005, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [60/100], Loss: 0.0004, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [70/100], Loss: 0.0002, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [80/100], Loss: 0.0001, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [90/100], Loss: 0.0001, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [100/100], Loss: 0.0000, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000


In [103]:

# evaluate the model
model.eval()
with torch.no_grad():
    y_hat = model(X_test_tensor)
    _, predicted = torch.max(y_hat.data, 1)
    # accuracy is the number of correct predictions divided by the total number of predictions
    accuracy = (predicted == y_test_tensor).sum().item() / len(y_test_tensor)
    # precision is the number of true positives divided by the number of true positives plus the number of false positives
    precision = (predicted[y_test_tensor == 1] == 1).sum().item() / (predicted == 1).sum().item()
    # recall is the number of true positives divided by the number of true positives plus the number of false negatives
    recall = (predicted[y_test_tensor == 1] == 1).sum().item() / (y_test_tensor == 1).sum().item()
    
    # print 10 samples
    for i in range(10):
        item_prob_values = y_hat[i].tolist()
        for j in range(len(item_prob_values)):
            item_prob_values[j] = round(item_prob_values[j], 2)
        real_label = y_test_tensor[i].item()
        print(f"Sample {i+1}: True label: {real_label}, Predicted label: {predicted[i].item()}, Probabilities: {item_prob_values}")

    print(f"Evaluation - Accuracy: {accuracy:.6f}, Precision: {precision:.6f}, Recall: {recall:.6f}")

Sample 1: True label: 8, Predicted label: 8, Probabilities: [-19.17, -16.37, -2.51, -8.84, -20.08, -24.52, -19.73, -17.26, 17.94, -25.79]
Sample 2: True label: 1, Predicted label: 1, Probabilities: [-17.12, 15.77, -6.18, -9.81, -11.88, -14.59, -13.35, -3.52, -4.6, -15.61]
Sample 3: True label: 9, Predicted label: 9, Probabilities: [-16.68, -31.94, -31.62, -11.67, -6.6, -12.3, -40.85, -4.9, -10.33, 22.13]
Sample 4: True label: 9, Predicted label: 9, Probabilities: [-22.62, -49.75, -34.7, -13.29, -2.77, -7.03, -40.76, 9.87, -11.69, 25.97]
Sample 5: True label: 8, Predicted label: 8, Probabilities: [-21.2, -20.64, -20.71, -8.57, -24.24, -13.53, -20.77, -33.87, 22.09, -16.65]
Sample 6: True label: 6, Predicted label: 6, Probabilities: [-3.52, -20.8, -7.51, -25.32, -7.72, 1.6, 22.04, -42.54, -2.73, -22.58]
Sample 7: True label: 2, Predicted label: 2, Probabilities: [-23.41, -29.58, 26.34, -3.24, -17.96, -33.98, -20.81, -31.15, -16.84, -8.48]
Sample 8: True label: 2, Predicted label: 2, Prob

In [104]:
# build sample submission file from the example submission file ./data/digit-recognizer/sample_submission.csv
sample_submission = pd.read_csv("../data/digit-recognizer/sample_submission.csv")
sample_submission.head()


,ImageId,Label
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [105]:
# ../data/digit-recognizer/test.csv
test_df = pd.read_csv("../data/digit-recognizer/test.csv")
test_df.head()

# make predictions for the test set
test_tensor = torch.tensor(test_df.values, dtype=torch.float32)
with torch.no_grad():
    y_hat = model(test_tensor)
    _, predicted = torch.max(y_hat.data, 1)
# create submission file
submission = pd.DataFrame(
    {"ImageId": range(1, len(predicted) + 1), "Label": predicted.numpy()}
)
submission.to_csv("../data/digit-recognizer/submission.csv", index=False)


In [106]:
# !kaggle competitions submit -c digit-recognizer -f ../data/digit-recognizer/submission.csv -m "Message"